# 🚦 Sidra Intersection – Worst Movement Summary Extractor

This notebook extracts key performance data from **SIDRA INTERSECTION** PDF reports and produces a clean Excel summary focused on the **worst movement** at each junction.

---

### 📋 Output Columns

| Column | Description |
|--------|-------------|
| Scenario | Time period / scenario name (e.g. Base AM, Base PM) |
| Location | Junction reference (e.g. Jn 1, Jn 2) |
| V/C | Volume-to-Capacity ratio of the worst movement |
| Delay | Average delay (sec) of the worst movement |
| LOS | Level of Service of the worst movement |
| Max Queue Length | 95th percentile back-of-queue length (m) |
| Approach | Approach direction of the worst movement |
| W_Movement | Worst movement turn description |
| W_Delay | Worst movement delay (sec) |
| W_V/C | Worst movement V/C ratio |
| W_LOS | Worst movement Level of Service |

---

### 🔢 LOS Thresholds (HCM)

| LOS | Delay (sec/veh) |
|-----|-----------------|
| A | ≤ 10 |
| B | 10 – 15 |
| C | 15 – 25 |
| D | 25 – 35 |
| E | 35 – 50 |
| F | > 50 |

---

### 📊 Sample Output

Below is an example of the kind of result this notebook produces:

| Scenario | Location | V/C | Delay | LOS | Max Queue Length | Approach | W_Movement | W_Delay | W_V/C | W_LOS |
|----------|----------|-----|-------|-----|-----------------|----------|------------|---------|-------|-------|
| Base AM | Jn 24 | 0.256 | 3.5 | A | 8.7 | South | South R2 | 3.8 | 0.256 | A |
| Base AM | Jn 25 | 0.336 | 4.2 | A | 13.0 | East | North L2 | 5.3 | 0.197 | A |
| Base AM | Jn 23 | 0.608 | 8.9 | A | 36.4 | South | East L2 | 10.7 | 0.556 | B |
| Base PM | Jn 1 | 3.241 | 230.8 | F | 599.2 | SouthEast | SouthEast L2 | 1112.6 | 3.241 | F |

> **Note:** Junctions are sorted in numeric order (Jn 1 → Jn 2 → … → Jn 10 → Jn 11) within each scenario, regardless of any suffix or prefix in the junction name (e.g. `Jn 34v`, `Jn 4 - RIRO`).

---

## Step 1 – Install Required Libraries

Run this cell **once** per Colab session. It installs:
- **pdfplumber** – extracts text and tables from PDF pages
- **pymupdf** – splits multi-page PDFs into individual pages

In [ ]:
!pip install pdfplumber pymupdf --quiet
print('✅ Libraries installed.')

## Step 2 – Upload Sidra PDF Report(s)

A file picker will appear. Select **one or more** Sidra PDF reports to process.

> You can upload multiple PDFs at once (e.g. Base AM and Base PM reports).

In [ ]:
from google.colab import files

print('Please select your Sidra PDF file(s) to upload...')
uploaded = files.upload()

print(f'\n✅ {len(uploaded)} file(s) uploaded successfully:')
for name in uploaded.keys():
    print(f'   • {name}')

## Step 3 – Import Libraries & Define Helper Functions

Imports all required libraries and defines two helper functions:
- **`categorize_value()`** – converts the V/C ratio (degree of saturation) into an LOS letter using the B17 thresholds from the original Sidra workflow
- **`natural_sort_key()`** – sorts filenames numerically (so `Sidra_JN_2` comes before `Sidra_JN_10`)
- **`junction_sort_key()`** – extracts the junction number from any Location string for correct numeric ordering in the final output

In [ ]:
from collections import namedtuple
import pandas as pd
import pdfplumber
import fitz          # pymupdf
import re
import os
import shutil
import warnings
import logging

# Suppress noisy warnings from PDF libraries
warnings.filterwarnings('ignore')
logging.getLogger('pdfminer').setLevel(logging.ERROR)


def categorize_value(B17: float) -> str:
    """
    Convert V/C ratio (degree of saturation) to HCM Level of Service letter.
    Used for individual movement and approach LOS assignments.
    """
    if B17 <= 10:          return 'A'
    elif 10 < B17 <= 15:   return 'B'
    elif 15 < B17 <= 25:   return 'C'
    elif 25 < B17 <= 35:   return 'D'
    elif 35 < B17 <= 50:   return 'E'
    else:                  return 'F'


def natural_sort_key(filename: str) -> list:
    """
    Natural sort: extracts all integers from a filename so that
    Sidra_JN_2.pdf sorts before Sidra_JN_10.pdf.
    """
    return [int(n) for n in re.findall(r'\d+', filename)]


def junction_sort_key(location: str) -> int:
    """
    Extracts the leading junction number from any Location string.
    Works with suffixes and prefixes:  'Jn 4 - RIRO' -> 4
                                       'Jn 34v'       -> 34
                                       'Junction 10'  -> 10
    """
    m = re.search(r'(\d+)', str(location))
    return int(m.group(1)) if m else 0


print('✅ Libraries imported and helper functions defined.')

## Step 4 – Split PDFs into Individual MOVEMENT SUMMARY Pages

Each Sidra PDF may contain multiple junctions. This step:
1. Scans every page of each uploaded PDF
2. Identifies pages that start with a **MOVEMENT SUMMARY** header
3. Saves each such page as a separate single-page PDF in the `output_pages/` folder

Splitting first makes parsing each junction independently much more reliable.

In [ ]:
OUTPUT_FOLDER = 'output_pages'

# Clean up any previous run
if os.path.exists(OUTPUT_FOLDER):
    shutil.rmtree(OUTPUT_FOLDER)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

count = 0
for filename in uploaded.keys():
    doc = fitz.open(filename)
    pages_found = 0
    for i, page in enumerate(doc):
        text  = page.get_text('text')
        lines = text.splitlines()
        if lines and 'MOVEMENT SUMMARY' in lines[0]:
            new_doc = fitz.open()
            new_doc.insert_pdf(doc, from_page=i, to_page=i)
            out_path = os.path.join(OUTPUT_FOLDER, f'Sidra_JN_{count + 1}.pdf')
            new_doc.save(out_path)
            count       += 1
            pages_found += 1
    print(f'   📄 {filename}  →  {pages_found} MOVEMENT SUMMARY page(s) found')

print(f'\n✅ Total pages extracted: {count}')

## Step 5 – Parse Each Page and Extract Worst Movement Data

For every extracted page this step:
- Reads the **Scenario** and **Location** from the page header
  - Handles all junction name formats: `Jn 1`, `Jn 1v`, `Jn 34 - RIRO`, `Jn 34v`, `Junction 10`
- Extracts the table of vehicle movements and their V/C, Delay, LOS and Queue values
- Identifies the **worst movement** as the one with the highest V/C ratio (degree of saturation)
- When multiple movements share the same maximum V/C, picks the one with the **highest delay**
- Records the approach direction, movement turn, delay, V/C and LOS for the worst movement

> **Sorting:** Results are ordered by Scenario (in the order they appear in the PDF) then by junction number numerically — so Jn 1 → Jn 2 → Jn 3 … Jn 10 → Jn 11, regardless of suffix or prefix.

In [ ]:
Output = pd.DataFrame()

# ── Use natural sort so Sidra_JN_2.pdf comes before Sidra_JN_10.pdf ──────
pdf_files = sorted(
    [f for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.pdf')],
    key=natural_sort_key
)

for pdf_file in pdf_files:
    pdf_path = os.path.join(OUTPUT_FOLDER, pdf_file)

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue

            lines = text.splitlines()

            # ── 1. Extract Scenario and Location from the page header ─────
            # Improved regex: uses \S+ for site number to catch '34v', '1v' etc.
            scenario   = None
            tmc_number = None
            for line in lines:
                m = re.search(
                    r'Site:\s*\S+\s*\[(.+?)\s*\(Site Folder:\s*(.+?)\)\]', line
                )
                if m:
                    tmc_number = m.group(1).strip()   # e.g. 'Jn 1', 'Jn 34v'
                    scenario   = m.group(2).strip()   # e.g. 'Base AM'
                    break
            if not scenario:
                continue

            # ── 2. Extract the data table ─────────────────────────────────
            table = page.extract_table()
            if not table:
                continue

            # Find the header row (contains 'Deg.' for degree of saturation)
            header_idx = None
            for idx, row in enumerate(table):
                if row and any('Deg.' in str(cell) for cell in row if cell):
                    header_idx = idx
                    break
            if header_idx is None:
                continue

            # ── 3. Parse movement rows to find the worst movement ─────────
            # Approach direction tracker
            approach_pattern = re.compile(
                r'^(North(?:East|West)?|South(?:East|West)?|East|West|North|South)\s*:',
                re.IGNORECASE
            )

            Que         = pd.DataFrame()   # holds all individual movement rows
            current_app = None

            for row in table[header_idx + 1:]:
                if not row or all(c is None or str(c).strip() == '' for c in row):
                    continue

                row_text = ' '.join(str(c) for c in row if c)

                # Detect approach direction header
                am = approach_pattern.match(row_text.strip())
                if am:
                    current_app = am.group(1).strip()
                    continue

                # Skip summary rows (Approach / All Vehicles)
                first_cell = next((str(c).strip() for c in row if c and str(c).strip()), '')
                if first_cell in ('Approach', 'All Vehicles') or first_cell == '':
                    continue

                # Try to parse a movement data row
                # Columns: ID | Turn/Mov | Class | v/c | delay | LOS | Queue | ...
                try:
                    cells = [str(c).strip() if c else '' for c in row]

                    # Find numeric values in the row
                    nums = []
                    for c in cells:
                        try:
                            nums.append(float(c))
                        except ValueError:
                            nums.append(None)

                    # Locate v/c (Deg. Satn), delay, LOS, queue columns
                    # Standard Sidra layout (0-indexed after stripping empties):
                    # 0:ID  1:Turn  2:Class  3:demand  4:HV  5:arr  6:HV  7:v/c  8:delay  9:LOS  10:queue_veh  11:queue_m ...
                    non_empty = [c for c in cells if c != '']
                    if len(non_empty) < 8:
                        continue

                    # Extract turn movement label (2nd non-empty cell)
                    turn_mov = non_empty[1] if len(non_empty) > 1 else ''

                    # Skip class rows and header fragments
                    if 'All MCs' not in row_text and 'HV' not in row_text:
                        if not any(c in row_text for c in ['LOS', 'v/c']):
                            pass  # continue parsing

                    # Pull numeric values robustly
                    num_vals = [v for v in nums if v is not None]
                    if len(num_vals) < 6:
                        continue

                    vc_val    = num_vals[4] if len(num_vals) > 4 else None
                    delay_val = num_vals[5] if len(num_vals) > 5 else None
                    queue_val = num_vals[7] if len(num_vals) > 7 else None

                    # LOS from row text
                    los_match = re.search(r'LOS\s+([A-F])', row_text)
                    los_val   = los_match.group(1) if los_match else (
                        categorize_value(delay_val) if delay_val else None
                    )

                    if vc_val is None or delay_val is None:
                        continue

                    row_df = pd.DataFrame([{
                        'Approach':        current_app,
                        'Turn':            turn_mov,
                        'v/c':             vc_val,
                        'Delay':           delay_val,
                        'LOS':             los_val,
                        'Queue_m':         queue_val,
                    }])
                    Que = pd.concat([Que, row_df], ignore_index=True)

                except Exception:
                    continue

            if Que.empty:
                continue

            # ── 4. Identify overall worst movement (max v/c, tie-break: max delay) ─
            Que['v/c']   = pd.to_numeric(Que['v/c'],   errors='coerce')
            Que['Delay'] = pd.to_numeric(Que['Delay'], errors='coerce')

            max_vc    = Que['v/c'].max()
            max_vc_df = Que[Que['v/c'] == max_vc]

            if len(max_vc_df) > 1:
                # Tie-break: highest delay
                best_idx = max_vc_df['Delay'].idxmax()
            else:
                best_idx = max_vc_df.index[0]

            worst = Que.loc[best_idx]

            w_vc       = worst['v/c']
            w_delay    = worst['Delay']
            w_los      = categorize_value(w_delay) if w_delay else worst['LOS']
            w_approach = worst['Approach']
            w_movement = f"{worst['Approach']} {worst['Turn']}"
            w_queue    = worst['Queue_m']

            new_row = {
                'Scenario':         scenario,
                'Location':         tmc_number,
                'V/C':              w_vc,
                'Delay':            w_delay,
                'LOS':              w_los,
                'Max Queue Length': w_queue,
                'Approach':         w_approach,
                'W_Movement':       w_movement,
                'W_Delay':          w_delay,
                'W_V/C':            w_vc,
                'W_LOS':            w_los,
            }

            Output = pd.concat([Output, pd.DataFrame([new_row])], ignore_index=True)

# ── 5. Sort output: Scenario order preserved, junction numbers numeric ────
if not Output.empty:
    Output['_jn_num']     = Output['Location'].apply(junction_sort_key)
    scenario_order        = Output['Scenario'].unique().tolist()
    Output['_scen_order'] = Output['Scenario'].apply(lambda s: scenario_order.index(s))
    Output = (
        Output
        .sort_values(['_scen_order', '_jn_num'])
        .drop(columns=['_jn_num', '_scen_order'])
        .reset_index(drop=True)
    )

warnings.filterwarnings('ignore')
logging.getLogger('pdfminer').setLevel(logging.ERROR)

print(f'✅ Parsed {len(Output)} junction(s) successfully.')
print(f'   Scenarios  : {Output["Scenario"].unique().tolist()}')
print(f'   Order check (first 5): {Output["Location"].head().tolist()}\n')
Output

## Step 6 – Export Results to Excel

Saves the summary table to **Sidra_Worst_Movement_Report.xlsx** and downloads it automatically.

In [ ]:
OUTPUT_FILE = 'Sidra_Worst_Movement_Report.xlsx'

Output.to_excel(OUTPUT_FILE, index=False)
print(f'✅ Report saved as "{OUTPUT_FILE}"')
print(f'   Rows    : {len(Output)}')
print(f'   Columns : {list(Output.columns)}')

# Automatically download the file
files.download(OUTPUT_FILE)
print('\n⬇️  Download started.')

---
## ✅ Done!

Your **Sidra_Worst_Movement_Report.xlsx** has been downloaded.

---

### 🔧 Troubleshooting

| Issue | Likely Cause | Fix |
|-------|-------------|-----|
| Empty output | PDF pages don't start with `MOVEMENT SUMMARY` | Check PDF was exported directly from SIDRA (not scanned) |
| Missing junctions | Site header format is unusual | Check the line containing `Site Folder:` on that page |
| Junction out of order | Location contains no number | Verify the Location value in the output — it will sort to position 0 |
| Wrong delay / V/C | Table columns shifted | May occur if the PDF has a non-standard Sidra export format |

### 📁 Compatible Junction Name Formats

The extractor correctly handles all of the following:

| Format | Example |
|--------|---------|
| Standard | `Jn 1`, `Jn 10` |
| With suffix | `Jn 1v`, `Jn 34v` |
| With descriptor | `Jn 4 - RIRO`, `Jn 10 - RIRO` |
| Full word | `Junction 1`, `Junction 10` |
| Combined | `Jn 34v - RIRO` |